In [1]:
from pathlib import Path
import pandas as pd
pd.set_option('max_colwidth', 100)
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix
import itertools
import scipy.stats as st
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 500)
import pingouin as pg
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.feature_selection import mutual_info_classif
import random
from random import randint
from contextlib import redirect_stdout

# Import data

In [2]:
seed = 12345
rng = np.random.default_rng(seed=seed)
# paths where to load files from
dat_dir = Path('../data/')
val_dir = dat_dir / 'ValSample'
fin_dir = dat_dir / 'finaldata'
log_dir = Path('./log/logs_final/logs_convdiv')
log_dir.mkdir(exist_ok=True, parents=True)
convdiv_res_dir = dat_dir / 'convdiv'
convdiv_res_dir.mkdir(exist_ok=True)
# path where data are saved
path_save_dat_gp_grid1st_full = fin_dir / 'dat_gp_grid1st_full.csv'
path_save_dat_en_grid1st_full = fin_dir / 'dat_en_grid1st_full.csv'

# load from csv
data_gp = pd.read_csv(path_save_dat_gp_grid1st_full)
data_en = pd.read_csv(path_save_dat_en_grid1st_full)

# Super mild processing

# create subject id column
data_gp = data_gp.rename(columns={'Unnamed: 0': 'Subject'})
data_en = data_en.rename(columns={'Unnamed: 0': 'Subject'})

print(data_gp.shape)
print(data_en.shape)

# combine
data_combined = pd.concat([data_gp, data_en])

print(data_combined.shape)

(497, 804)
(305, 804)
(802, 804)


In [3]:
# check stuff
data_combined.head(3)

,Subject,caseid,weight,sample_type,screener_1ongoing,screener_2impact,screener_3depression,screener_4anxiety,screener_5attention,consent,mood_yn,FNM_Q25_1,FNM_Q25_2,FNM_Q25_3,FNM_Q25_4,FNM_Q25_5,FNM_Q25_6,FNM_Q25_955,FNM_Q25_933,FNM_Q25_open1,mood_years,FNM_Q27_1,FNM_Q27_2,FNM_Q27_3,FNM_Q27_4,FNM_Q27_5,FNM_Q27_6,FNM_Q27_7,FNM_Q27_8,FNM_Q27_9,FNM_Q27_10,FNM_Q27_11,FNM_Q27_12,FNM_Q27_13,FNM_Q27_14,FNM_Q27_15,FNM_Q27_16,FNM_Q27_17,FNM_Q27_18,FNM_Q27_19,FNM_Q27_955,FNM_Q27_933,FNM_Q27_open1,mood_bothered_orig,anxiety_yn,FNM_Q30_m_1,FNM_Q30_m_2,FNM_Q30_m_3,FNM_Q30_m_4,FNM_Q30_m_5,FNM_Q30_m_6,FNM_Q30_m_7,FNM_Q30_m_8,FNM_Q30_m_955,FNM_Q30_m_933,FNM_Q30_open1,anxiety_years,FNM_Q32_1,FNM_Q32_2,FNM_Q32_3,FNM_Q32_4,FNM_Q32_5,FNM_Q32_6,FNM_Q32_7,FNM_Q32_8,FNM_Q32_9,FNM_Q32_10,FNM_Q32_955,FNM_Q32_933,FNM_Q32_open1,anxiety_bothered_orig,attention_yn,FNM_Q35_m_1,FNM_Q35_m_2,FNM_Q35_m_3,FNM_Q35_m_933,FNM_Q35_open1,attention_years,FNM_Q37_m_1,FNM_Q37_m_2,FNM_Q37_m_3,FNM_Q37_m_4,FNM_Q37_m_5,FNM_Q37_m_6,FNM_Q37_m_7,FNM_Q37_m_8,FNM_Q37_m_9,FNM_Q37_m_955,FNM_Q37_m_933,FNM_Q37_open1,attention_bothered_orig,inattention_1,inattention_2,inattention_3,inattention_4,inattention_5,inattention_6,FNM_Q1_attn,inattention_7,inattention_8,inattention_9,hyperactivity_1,hyperactivity_2,hyperactivity_3,hyperactivity_4,hyperactivity_5,impulsivity_1,impulsivity_2,impulsivity_3,impulsivity_4,sct_1,sct_2,sct_3,sct_4,sct_5,sct_6,sct_7,sct_8,sct_9,gad_1,gad_2,gad_3,gad_4,gad_5,gad_6,gad_7,phq_1,phq_2,phq_3,phq_4,phq_5,phq_6,phq_7,phq_8,hitop157,hitop81,hitop34,hitop54,hitop243,hitop182,hitop69,hitop89,hitop50,check_moderately,hitop129,hitop265,hitop124,hitop231,hitop93,hitop67,hitop245,hitop281,hitop141,hitop40,hitop204,hitop21,hitop236,hitop280,hitop84,hitop120,hitop77,hitop92,hitop258,hitop39,hitop254,hitop215,hitop95,hitop106,hitop283,hitop16,hitop20,hitop189,hitop1,hitop136,hitop246,hitop248,hitop257,hitop114,hitop117,hitop250,hitop200,hitop160,hitop23,hitop165,hitop244,hitop9,hitop142,hitop230,hitop149,hitop247,hitop99,hitop66,hitop240,hitop222,hitop90,hitop113,hitop278,hitop203,hitop159,hitop123,hitop275,hitop268,hitop225,hitop143,hitop151,hitop181,hitop211,hitop17,hitop126,hitop5,hitop261,hitop220,check_notatall,hitop15,hitop72,hitop140,hitop109,hitop197,hitop104,todayinattention_1,todayinattention_2,todayinattention_3,todayinattention_4,todayinattention_5,todayinattention_6,todayinattention_7,todayinattention_8,todayinattention_9,todayhyperactivity_1,todayhyperactivity_2,todayhyperactivity_3,todayhyperactivity_4,todayhyperactivity_5,todayimpulsivity_1,todayimpulsivity_2,todayimpulsivity_3,todayimpulsivity_4,todaysct_1,todaysct_2,todaysct_3,todaysct_4,todaysct_5,todaysct_6,todaysct_7,todaysct_8,todaysct_9,today_na1,todaygad_1,todaygad_2,todaygad_3,...,hitop165_recontact,hitop244_recontact,hitop9_recontact,hitop142_recontact,hitop230_recontact,hitop149_recontact,hitop247_recontact,hitop99_recontact,hitop66_recontact,hitop240_recontact,hitop222_recontact,hitop90_recontact,hitop113_recontact,hitop278_recontact,hitop203_recontact,hitop159_recontact,hitop123_recontact,hitop275_recontact,hitop268_recontact,hitop225_recontact,hitop143_recontact,hitop151_recontact,hitop181_recontact,hitop211_recontact,hitop17_recontact,hitop126_recontact,hitop5_recontact,hitop261_recontact,hitop220_recontact,check_notatall_recontact,hitop15_recontact,hitop72_recontact,hitop140_recontact,hitop109_recontact,hitop197_recontact,hitop104_recontact,todayinattention_1_recontact,todayinattention_2_recontact,todayinattention_3_recontact,todayinattention_4_recontact,todayinattention_5_recontact,todayinattention_6_recontact,todayinattention_7_recontact,todayinattention_8_recontact,todayinattention_9_recontact,todayhyperactivity_1_recontact,todayhyperactivity_2_recontact,todayhyperactivity_3_recontact,todayhyperactivity_4_recontact,todayhyperactivity_5_recontact,todayimpulsivity_1_recontact,todayimpulsivity_2_recontact,todayimpulsivity_3_recontact,todayimpulsivity_4_recontact,todaysct_1_recontact,

## Replace the OG hitop measures with invariant cores

Each HiTOP scale is resolved to its core sum-score column **programmatically** from `cores.pkl` (no hardcoded `__invcore` / `__ex_invcore` / full-scale name lists). Resolution priority per scale: confirmatory stepwise core (`hitop_{scale}__invcore`), then full-scale confirmatory core (bare `hitop_{scale}` column), then exhaustive alternate (`hitop_{scale}__ex_invcore`), then — if no invariant core exists at all — the full scale with a printed warning (the prereg's rule for such scales is to test the relevant hypotheses in the enriched sample only). Every resolved scale gets a canonical alias column `hitop_{scale}__core`, and all hypothesis code below uses only those aliases, so nothing downstream depends on which suffix a scale happened to earn.

The resolution table (printed below) shows each scale's source column, resolution path, and `core_level` — remember the reporting rule: clinical-vs-HV latent **mean comparisons** may only use `core_level == 'scalar'` scales; the convergent/divergent correlations here may use metric fallbacks.

In [4]:
icc_dir = dat_dir / 'icc'

cores = pd.read_pickle(icc_dir / 'cores.pkl')
#defrag datasets
data_gp = data_gp.copy()
data_en = data_en.copy()
data_combined = data_combined.copy()
# Sum scores for the REDUCED cores (measEq pipeline: cores.pkl carries a
# 'reduced' flag; full-scale cores keep the bare hitop_{scale} columns that
# already exist in the data). core_level ('scalar' vs 'metric' fallback) is
# also carried on each row.
for data in data_gp, data_en, data_combined:
    for row in cores.loc[cores.reduced, :].itertuples():
        tmp_items = list(row.item_nos)
        tmp_items_recontact = [ti + '_recontact' for ti in tmp_items]
        data[row.scale] = data.loc[:, tmp_items].sum(axis=1)
        data[f'{row.scale}_recontact'] = data.loc[:, tmp_items_recontact].sum(axis=1)

In [5]:
cores

,scale,core_level,item_nos,removed_nos,items,removed,error,confirmatory,core_set,inv,reduced
0,hitop_anhedonic_depression__invcore,scalar,"[hitop39, hitop123, hitop182, hitop230, hitop246]","[hitop77, hitop84, hitop92, hitop93, hitop157]","[It felt like there wasn’t anything interesting or fun to do., Nothing made me laugh., I was una...","[I didn’t look forward to seeing friends or family., I felt depressed., It took a lot of effort ...",None,True,gp_en,True,True
1,hitop_anxious_worry__invcore,scalar,"[hitop34, hitop89, hitop240, hitop248, hitop265]","[hitop20, hitop203]","[Thoughts were racing through my head., I had a lot of nervous energy., I felt nervous and ""on e...","[I felt tense., I felt very stressed.]",None,True,gp_en,True,True
2,hitop_appetite_gain__invcore,scalar,"[hitop120, hitop243, hitop275]",[hitop141],"[I could not keep myself from eating., I stuffed myself with food., I ate even when I was not re...",[I thought a lot about food.],None,True,gp_en,True,True
3,appetite_loss,scalar,"[hitop280, hitop283, hitop109]",[],"[My appetite was poor., I lost a significant amount of weight without even trying., I did not fe...",[],None,True,gp_en,True,False
4,cognitive_problems,scalar,"[hitop67, hitop159, hitop189, hitop142]",[],"[I could not stay focused on what I was doing., I was unable to keep my mind on what I was doing...",[],None,True,gp_en,True,False
5,hitop_hyposomnia__invcore,scalar,"[hitop99, hitop5, hitop66]","[hitop181, hitop231]","[I needed much less sleep than usual., I had days when I never got tired., I did not feel tired,...","[I felt like I could keep going and going without ever getting tired., I felt like I could go fo...",None,True,gp_en,True,True
6,indecisiveness,scalar,"[hitop21, hitop90, hitop95]",[],"[I was indecisive., It was difficult for me to make decisions., I had trouble making up my mind.]",[],None,True,gp_en,True,False
7,hitop_insomnia__invcore,scalar,"[hitop160, hitop254, hitop268]",[hitop261],"[I had trouble staying asleep., I slept very poorly., I lay awake for a long time before falling...",[I woke up early and could not get back to sleep.],None,True,gp_en,True,True
8,panic,scalar,"[hitop15, hitop104, hitop126, hitop211, hitop215, hitop257]",[],"[I was short of breath., I felt nauseated., My heart was racing or pounding., I was trembling or...",[],None,True,gp_en,True,False
9,separation_insecurity,scalar,"[hitop40, hitop50, hitop69, hitop81, hitop113, hitop136, hitop151, hitop197]",[],"[I felt insecure about important relationships in my life., I wanted someone else to make decisi...",[],None,True,gp_en,True,False


In [6]:
# ---- Resolve every HiTOP scale to its core sum-score column (GENERATED;
# ---- previously hardcoded lists of __invcore/__ex_invcore/full-scale names) ----

# Substantive domain groupings (theory, not analysis outcomes): which HiTOP
# scales count as depression vs anxiety. well_being is in neither and is
# excluded from the HiTOP sum per the prereg.
DEP_SCALES = ['anhedonic_depression', 'appetite_gain', 'appetite_loss',
              'cognitive_problems', 'hyposomnia', 'indecisiveness',
              'insomnia', 'shame_guilt']
ANX_SCALES = ['anxious_worry', 'panic', 'separation_insecurity',
              'situational_phobia', 'social_anxiety']
HITOP_SUM_SCALES = DEP_SCALES + ANX_SCALES          # excludes well_being
ALL_HITOP_SCALES = HITOP_SUM_SCALES + ['well_being']

core_names = set(cores.scale)
core_level_by_name = dict(zip(cores.scale, cores.core_level))


def resolve_core_col(scale):
    """Column holding this scale's core sum score, plus how it was resolved.

    Priority: confirmatory stepwise core > full-scale confirmatory core >
    exhaustive alternate > full scale with no invariant core (prereg:
    enriched-sample-only hypotheses for these)."""
    invcore = f'hitop_{scale}__invcore'
    ex_invcore = f'hitop_{scale}__ex_invcore'
    if invcore in core_names:
        return invcore, 'confirmatory_reduced', core_level_by_name[invcore]
    if scale in core_names:  # full-scale core rows keep the bare scale name
        return f'hitop_{scale}', 'confirmatory_full_scale', core_level_by_name[scale]
    if ex_invcore in core_names:
        return ex_invcore, 'exhaustive_alternate', core_level_by_name[ex_invcore]
    return f'hitop_{scale}', 'no_invariant_core', None


core_resolution = {}
for scale in ALL_HITOP_SCALES:
    col, how, level = resolve_core_col(scale)
    core_resolution[scale] = dict(column=col, resolution=how, core_level=level)

core_resolution_df = pd.DataFrame(core_resolution).T
print(core_resolution_df.to_string())

no_core = core_resolution_df.index[
    core_resolution_df.resolution == 'no_invariant_core'].tolist()
if no_core:
    print(f"\nWARNING: no invariant core for {no_core} -- the full scale is "
          f"used; per the prereg, hypotheses involving these scales should "
          f"be tested in the enriched sample only.")

# canonical alias columns: all hypothesis code below uses hitop_{scale}__core
for data in (data_gp, data_en, data_combined):
    for scale, info in core_resolution.items():
        data[f'hitop_{scale}__core'] = data[info['column']]

# resolved column lists (previously hardcoded)
dep_scales = [f'hitop_{s}__core' for s in DEP_SCALES]
anx_scales = [f'hitop_{s}__core' for s in ANX_SCALES]
hitop_sum_cols = [f'hitop_{s}__core' for s in HITOP_SUM_SCALES]

                                                    column               resolution core_level
anhedonic_depression   hitop_anhedonic_depression__invcore     confirmatory_reduced     scalar
appetite_gain                 hitop_appetite_gain__invcore     confirmatory_reduced     scalar
appetite_loss                          hitop_appetite_loss  confirmatory_full_scale     scalar
cognitive_problems                hitop_cognitive_problems  confirmatory_full_scale     scalar
hyposomnia                       hitop_hyposomnia__invcore     confirmatory_reduced     scalar
indecisiveness                        hitop_indecisiveness  confirmatory_full_scale     scalar
insomnia                           hitop_insomnia__invcore     confirmatory_reduced     scalar
shame_guilt                     hitop_shame_guilt__invcore     confirmatory_reduced     scalar
anxious_worry                 hitop_anxious_worry__invcore     confirmatory_reduced     scalar
panic                                          hit

# Convergent validity hypotheses

In [7]:
# The p-value helps us determine whether or not we can meaningfully conclude that 
# the population correlation coefficient is different from zero, 
# based on what we observe from the sample.

"We will use these invariant scales in our subsequent tests of convergent and divergent validity. 

For scales in which we cannot establish an invariant core, we will test the relevant hypotheses in just the enriched sample."

In [8]:
def do_correlation(col1, col2):
    corr_result = pg.corr(col1, col2, method = 'spearman')
    #print(corr_result)
    my_r = corr_result.iloc[0]['r']
    my_r = "{:.2f}".format(float(my_r))
    my_ci = corr_result.iloc[0]['CI95']
    my_ci_1 = my_ci[0]
    my_ci_2 = my_ci[1]
    my_p = corr_result.iloc[0]['p_val']
    my_p = "{:.3f}".format(float(my_p)) 
    return(my_p, my_r, my_ci_1, my_ci_2)

In [9]:
# Claude updated version to record a results dataframe.
# All HiTOP scale columns are the canonical hitop_{scale}__core aliases
# resolved above -- no hardcoded __invcore/__ex_invcore names.
def check_convergent_hypotheses(mydata=data_combined, mydata_gp=data_gp, mydata_en=data_en):
    '''
    tests convergent hypotheses; for hitop scales we use inv cores when appropriate

    Returns
    -------
    pd.DataFrame
        One row per test with columns:
        hypothesis, test_type, variables, statistic, p_value,
        ci_lower, ci_upper, significant, supported
    '''

    results = []  # collect one dict per test

    def record_correlation(hypothesis, var1_name, var2_name, p, r, ci1, ci2,
                           expected_direction):
        '''Helper: store a correlation test result and determine support.'''
        significant = float(p) < 0.05
        if not significant:
            supported = False
        elif expected_direction == 'positive':
            supported = float(r) > 0
        elif expected_direction == 'negative':
            supported = float(r) < 0
        else:  # any direction
            supported = True
        results.append({
            'hypothesis': hypothesis,
            'test_type': 'spearman_correlation',
            'variables': f'{var1_name} ~ {var2_name}',
            'statistic': float(r),
            'p_value': float(p),
            'ci_lower': float(ci1),
            'ci_upper': float(ci2),
            'significant': significant,
            'supported': supported,
        })
        return significant, supported

    print('\n --- CONV ANALYSIS START --- \n')

    print('\n-----> HYPOTHESIS C1 <-----\n')
    # HYPOTHESIS C1: hitop_anhedonic_depression will be positively correlated with PHQ-8
    # Spearman; alpha = .05
    (p, r, ci1, ci2) = do_correlation(mydata.hitop_anhedonic_depression__core, mydata.phq_sum)
    sig, sup = record_correlation('C1', 'hitop_anhedonic_depression__core', 'phq_sum',
                                  p, r, ci1, ci2, expected_direction='positive')
    if not sig:
        print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
    else:
        print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        if sup:
            print('Hyp C1 SUPPORTED: Anhedonic depression (or invariant core) positively correlates with PHQ-8')
        else:
            print('Hyp C1 NOT SUPPORTED: Anhedonic depression (or invariant core) DOES NOT positively correlate with PHQ-8')

    print('\n-----> HYPOTHESIS C2 <-----\n')
    # HYPOTHESIS C2: hitop_well-being will be negatively correlated with PHQ-8
    # Spearman; alpha = .05
    (p, r, ci1, ci2) = do_correlation(mydata.hitop_well_being__core, mydata.phq_sum)
    sig, sup = record_correlation('C2', 'hitop_well_being__core', 'phq_sum',
                                  p, r, ci1, ci2, expected_direction='negative')
    if not sig:
        print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
    else:
        print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        if sup:
            print('Hyp C2 SUPPORTED: Well-being (or invariant core) negatively correlates with PHQ-8')
        else:
            print('Hyp C2 NOT SUPPORTED: Well-being (or invariant core) DOES NOT negatively correlate with PHQ-8')

    print('\n-----> HYPOTHESIS C3 <-----\n')
    # HYPOTHESIS C3: hitop_anxious_worry will be positively correlated with GAD-7
    # Spearman; alpha = .05
    (p, r, ci1, ci2) = do_correlation(mydata.hitop_anxious_worry__core, mydata.gad_sum)
    sig, sup = record_correlation('C3', 'hitop_anxious_worry__core', 'gad_sum',
                                  p, r, ci1, ci2, expected_direction='positive')
    if not sig:
        print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
    else:
        print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        if sup:
            print('Hyp C3 SUPPORTED: Anxious worry (or invariant core) positively correlates with GAD-7')
        else:
            print('Hyp C3 NOT SUPPORTED: Anxious worry (or invariant core) DOES NOT positively correlate with GAD-7')

    print('\n-----> HYPOTHESIS C4 <-----\n')
    # HYPOTHESIS C4: Higher sum of scores on the HiTOP Appetite Gain and Appetite Loss
    # will be correlated with responses to PHQ-8 "Poor appetite or overeating"
    mydata['hitop_appetite_sum'] = mydata['hitop_appetite_gain__core'] + mydata['hitop_appetite_loss__core']
    (p, r, ci1, ci2) = do_correlation(mydata.hitop_appetite_sum, mydata.phq_5)
    sig, sup = record_correlation('C4', 'hitop_appetite_sum', 'phq_5',
                                  p, r, ci1, ci2, expected_direction='positive')
    if not sig:
        print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        print('Hyp C4 NOT SUPPORTED: Sum of scores on Appetite Gain and Appetite Loss (or invariant cores) DOES NOT correlate with PHQ item 5 - poor appetite or overeating')
    else:
        if sup:
            print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
            print('Hyp C4 SUPPORTED: Sum of scores on Appetite Gain and Appetite Loss (or invariant cores) is correlated with PHQ item 5 - poor appetite or overeating')

    print('\n-----> HYPOTHESIS C5 <-----\n')
    # HYPOTHESIS C5: the HiTOP Cognitive Problems will be positively correlated with responses
    # to the PHQ-8 item "Trouble concentrating on things, such as school work, reading or watching television".
    (p, r, ci1, ci2) = do_correlation(mydata.hitop_cognitive_problems__core, mydata.phq_7)
    sig, sup = record_correlation('C5', 'hitop_cognitive_problems__core', 'phq_7',
                                  p, r, ci1, ci2, expected_direction='positive')
    if not sig:
        print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
    else:
        print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        if sup:
            print('Hyp C5 SUPPORTED: HiTOP Cognitive problems positive correlates with PHQ-8 item 7 "Trouble concentrating on things..."')
        else:
            print('Hyp C5 NOT SUPPORTED: HiTOP Cognitive problems DOES NOT positive correlate with PHQ-8 item 7 "Trouble concentrating on things..."')

    print('\n-----> HYPOTHESIS C6 <-----\n')
    # HYPOTHESIS C6: The HiTOP Insomnia subscale will be positively correlated with responses
    # to the PHQ-8 item "Trouble falling or staying asleep, or sleeping too much"
    (p, r, ci1, ci2) = do_correlation(mydata.hitop_insomnia__core, mydata.phq_3)
    sig, sup = record_correlation('C6', 'hitop_insomnia__core', 'phq_3',
                                  p, r, ci1, ci2, expected_direction='positive')
    if not sig:
        print(f"Correlation NOT Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
    else:
        print(f"Correlation Significant with p-val = {p} and r = {r}({ci1}, {ci2})")
        if sup:
            print('Hyp C6 SUPPORTED: HiTOP Insomnia positive correlates with PHQ-8 item 3 "Trouble falling or staying asleep, or sleeping too much"')
        else:
            print('Hyp C6 NOT SUPPORTED: HiTOP Insomnia DOES NOT positive correlate with PHQ-8 item 3 "Trouble falling or staying asleep, or sleeping too much"')

    print('\n-----> HYPOTHESIS C7 <-----\n')
    # HYPOTHESIS C7: Higher sum of all HiTOP subscales will be associated with greater likelihood
    # of being bothered by symptoms of a diagnosed mood or anxiety disorder.

    # HiTOP sum over the resolved cores (excludes well_being per the prereg;
    # hitop_sum_cols is generated in the core-resolution cell above)
    mydata['hitop_sum_new'] = mydata.loc[:, hitop_sum_cols].sum(axis=1)
    mydata['moodanxiety_bothered_int'] = mydata.moodanxiety_bothered.astype(int)
    # using an example from here: https://www.science.smith.edu/~jcrouser/SDS293/labs/lab4-py.html
    formula = 'moodanxiety_bothered_int ~ hitop_sum_new'
    model = smf.glm(formula=formula, data=mydata, family=sm.families.Binomial())
    result = model.fit()
    print(result)
    print(result.summary())
    print(result.pvalues)

    # extract coefficient (z-statistic used as the test statistic), p-value, and CI
    coef = result.params.iloc[1]
    z_stat = result.tvalues.iloc[1]
    p_val_hitop_sum_float = result.pvalues.iloc[1]
    conf_int = result.conf_int()
    significant_c7 = p_val_hitop_sum_float < 0.05
    results.append({
        'hypothesis': 'C7',
        'test_type': 'logistic_regression',
        'variables': formula,
        'statistic': float(z_stat),  # z-statistic for hitop_sum_new coefficient
        'p_value': float(p_val_hitop_sum_float),
        'ci_lower': float(conf_int.iloc[1, 0]),  # CI on the coefficient
        'ci_upper': float(conf_int.iloc[1, 1]),
        'significant': significant_c7,
        'supported': significant_c7 and coef > 0,
    })

    p_val_hitop_sum = format(p_val_hitop_sum_float, 'f')
    if float(p_val_hitop_sum) >= 0.05:
        print(f"\nHitop sum of items is NOT a significant predictor of mood_anxiety_bothered, with p-val = {p_val_hitop_sum}")
        print('Hyp C7 NOT SUPPORTED')
    else:
        print(f"\nHitop sum of items IS A SIGNIFICANT PREDICTOR of mood_anxiety_bothered, with p-val = {p_val_hitop_sum}")
        print('Hyp C7 SUPPORTED')

    print('some exploratory C7 tests')
    for sample_label, sample_query in [('C7_exploratory_sample1', 'sample_type==1'),
                                       ('C7_exploratory_sample2', 'sample_type==2')]:
        model = smf.glm(formula=formula, data=mydata.query(sample_query), family=sm.families.Binomial())
        result = model.fit()
        print(result)
        print(result.summary())
        print(result.pvalues)
        conf_int = result.conf_int()
        results.append({
            'hypothesis': sample_label,
            'test_type': 'logistic_regression',
            'variables': f'{formula} [{sample_query}]',
            'statistic': float(result.tvalues.iloc[1]),
            'p_value': float(result.pvalues.iloc[1]),
            'ci_lower': float(conf_int.iloc[1, 0]),
            'ci_upper': float(conf_int.iloc[1, 1]),
            'significant': result.pvalues.iloc[1] < 0.05,
            'supported': result.pvalues.iloc[1] < 0.05 and result.params.iloc[1] > 0,
        })

    print('\n --- CONV ANALYSIS DONE ---')

    results_df = pd.DataFrame(results)
    return results_df

In [10]:
conv_res = check_convergent_hypotheses()
conv_res.to_pickle(convdiv_res_dir / 'conv_res.pkl')


 --- CONV ANALYSIS START --- 


-----> HYPOTHESIS C1 <-----

Correlation Significant with p-val = 0.000 and r = 0.82(0.8, 0.84)
Hyp C1 SUPPORTED: Anhedonic depression (or invariant core) positively correlates with PHQ-8

-----> HYPOTHESIS C2 <-----

Correlation Significant with p-val = 0.000 and r = -0.53(-0.58, -0.48)
Hyp C2 SUPPORTED: Well-being (or invariant core) negatively correlates with PHQ-8

-----> HYPOTHESIS C3 <-----

Correlation Significant with p-val = 0.000 and r = 0.89(0.88, 0.91)
Hyp C3 SUPPORTED: Anxious worry (or invariant core) positively correlates with GAD-7

-----> HYPOTHESIS C4 <-----

Correlation Significant with p-val = 0.000 and r = 0.77(0.74, 0.8)
Hyp C4 SUPPORTED: Sum of scores on Appetite Gain and Appetite Loss (or invariant cores) is correlated with PHQ item 5 - poor appetite or overeating

-----> HYPOTHESIS C5 <-----

Correlation Significant with p-val = 0.000 and r = 0.81(0.79, 0.83)
Hyp C5 SUPPORTED: HiTOP Cognitive problems positive correlates with PH

In [11]:
with open(log_dir / "mylog_convergent.txt", "w") as f:
    with redirect_stdout(f):
        check_convergent_hypotheses()

# Divergent validity hypotheses

In [12]:
# This will need to SET THE SEED
random.seed(seed)
# test
for i in range(5):
    print(random.random())
# after kernel restart, this should be 
# 0.41661987254534116
# 0.010169169457068361
# 0.8252065092537432
# 0.2986398551995928
# 0.3684116894884757

0.41661987254534116
0.010169169457068361
0.8252065092537432
0.2986398551995928
0.3684116894884757


In [13]:
# dep_scales / anx_scales / hitop_sum_cols were generated in the
# core-resolution cell above (hitop_{scale}__core aliases); re-derive here
# so this section is self-contained if re-run after the resolution cell
dep_scales = [f'hitop_{s}__core' for s in DEP_SCALES]
anx_scales = [f'hitop_{s}__core' for s in ANX_SCALES]
hitop_sum_cols = [f'hitop_{s}__core' for s in HITOP_SUM_SCALES]

In [14]:
mydata = data_combined.reset_index(drop=True)
mydata['gad_phq_sum'] = mydata['gad_sum'] + mydata['phq_sum']
# HiTOP sums over the resolved core columns (generated lists; excludes
# well_being per the prereg)
mydata['hitop_sum_invcores'] = mydata.loc[:, hitop_sum_cols].sum(axis=1)
mydata['hitop_all_depression__core'] = mydata.loc[:, dep_scales].sum(axis=1)
mydata['hitop_all_anxiety__core'] = mydata.loc[:, anx_scales].sum(axis=1)
mydata['baars_sum_d3']  = mydata.baars_sum.copy() # doubling baars_sum because it needs to be permuted for d1, but not for d


In [15]:
cols_to_z = [
    'gad_phq_sum',
    'hitop_sum_invcores',
    'baars_sum',
    'baars_sum_d3',
]
cols_for_d = [
    'gad_phq_sum',
    'hitop_sum_invcores',
    'baars_sum',
    'baars_sum_d3',
    'moodanxiety_bothered',
    'mood_bothered',
    'anxiety_bothered',
    'attention_bothered',
    'hitop_all_depression__core',
    'hitop_all_anxiety__core',
    'hitop_anhedonic_depression__core',
    'hitop_anxious_worry__core'
]
permed_cols = [
    'gad_phq_sum',
    'baars_sum',
    'moodanxiety_bothered',
    'attention_bothered',
    'mood_bothered',
    'anxiety_bothered'
]
nonpermed_cols = [cc for cc in cols_for_d if cc not in permed_cols]
nperms = 1000

In [16]:
ogddat = mydata.loc[:, ['caseid', 'sample_type'] + cols_for_d].copy()
ddat =  mydata.loc[:, ['caseid', 'sample_type'] + cols_for_d].copy()
ddat['pid'] = -1

In [17]:
def calc_dhs(df,pid=0, random_state=2342393490, cols_to_z=None):

    if cols_to_z is None:
        cols_to_z = [
            'gad_phq_sum',
            'hitop_sum_invcores',
            'baars_sum',
            'baars_sum_d3',
            'hitop_all_depression__core',
            'hitop_all_anxiety__core'
        ]
    df = df.copy()
    for cc in cols_to_z:
        df[cc] = (df[cc] - df[cc].mean()) / df[cc].std()
    # -----------------------------------------------------------------------
    # D1: There will be a stronger correlation between the sum of HiTOP scales and the sum of GAD-7 and PHQ-8 scores 
    #     than with BAARS-IV total score    
    gad_phq_v_hitop = stats.spearmanr(df.gad_phq_sum, df.hitop_sum_invcores)[0]
    baars_v_hitop = stats.spearmanr(df.baars_sum, df.hitop_sum_invcores)[0]
    d1 = gad_phq_v_hitop - baars_v_hitop
    
    # -----------------------------------------------------------------------
    # D2: The sum of HiTOP scales will be a better predictor of being bothered by symptoms of a diagnosed anxiety disorder 
    # than of being bothered by symptoms of ADHD.
    hitop_v_mab = mutual_info_classif(df.hitop_sum_invcores.values.reshape((-1,1)), df.moodanxiety_bothered.values, random_state=random_state)[0]
    hitop_v_ab = mutual_info_classif(df.hitop_sum_invcores.values.reshape((-1,1)), df.attention_bothered.values, random_state=random_state)[0]
    d2 = hitop_v_mab - hitop_v_ab
    # -----------------------------------------------------------------------
    # D3: The BAARS-IV total score will be a better predictor of being bothered by symptoms of ADHD 
    # than of a diagnosed mood or anxiety disorder.
    baars_v_ab = mutual_info_classif(df.baars_sum_d3.values.reshape((-1,1)), df.attention_bothered.values, random_state=random_state)[0]
    baars_v_mab = mutual_info_classif(df.baars_sum_d3.values.reshape((-1,1)), df.moodanxiety_bothered.values, random_state=random_state)[0]
    d3 = baars_v_ab - baars_v_mab
    # -----------------------------------------------------------------------
    # D4: The sum of HiTOP mood disorder scales will be a better predictor of being bothered by symptoms 
    # of a diagnosed mood disorder than of being bothered by symptoms of a diagnosed anxiety disorder.
    hitopdep_v_mb = mutual_info_classif(df.hitop_all_depression__core.values.reshape((-1,1)), df.mood_bothered.values, random_state=random_state)[0]
    hitopdep_v_ab = mutual_info_classif(df.hitop_all_depression__core.values.reshape((-1,1)), df.anxiety_bothered.values, random_state=random_state)[0]
    d4 =  hitopdep_v_mb - hitopdep_v_ab
    # exploratory 4, just use the anhedonic depression core
    hitopanhdep_v_mb = mutual_info_classif(df.hitop_anhedonic_depression__core.values.reshape((-1,1)), df.mood_bothered.values, random_state=random_state)[0]
    hitopanhdep_v_ab = mutual_info_classif(df.hitop_anhedonic_depression__core.values.reshape((-1,1)), df.anxiety_bothered.values, random_state=random_state)[0]
    exd4 =  hitopanhdep_v_mb - hitopanhdep_v_ab    
    # -----------------------------------------------------------------------
    # D5: The sum of HiTOP anxiety disorder scales will be a better predictor of being bothered by symptoms 
    # of a diagnosed anxiety disorder than of being bothered by symptoms of a diagnosed mood disorder.
    hitopanx_v_ab = mutual_info_classif(df.hitop_all_anxiety__core.values.reshape((-1,1)), df.anxiety_bothered.values, random_state=random_state)[0]
    hitopanx_v_mb = mutual_info_classif(df.hitop_all_anxiety__core.values.reshape((-1,1)), df.mood_bothered.values, random_state=random_state)[0]
    d5 = hitopanx_v_ab - hitopanx_v_mb
    # exploratory 5, just use the anxious worry core
    hitopanxwor_v_ab = mutual_info_classif(df.hitop_anxious_worry__core.values.reshape((-1,1)), df.anxiety_bothered.values, random_state=random_state)[0]
    hitopanxwor_v_mb = mutual_info_classif(df.hitop_anxious_worry__core.values.reshape((-1,1)), df.mood_bothered.values, random_state=random_state)[0]
    exd5 =  hitopanxwor_v_ab - hitopanxwor_v_mb    

    res = dict(
        pid=pid,
        gad_phq_v_hitop=gad_phq_v_hitop,
        baars_v_hitop=baars_v_hitop,
        d1=d1,
        hitop_v_mab=hitop_v_mab,
        hitop_v_ab=hitop_v_ab,
        d2=d2,
        baars_v_ab=baars_v_ab,
        baars_v_mab=baars_v_mab,
        d3=d3,
        hitopdep_v_mb=hitopdep_v_mb,
        hitopdep_v_ab=hitopdep_v_ab,
        d4=d4,
        hitopanx_v_ab=hitopanx_v_ab,
        hitopanx_v_mb=hitopanx_v_mb,
        d5=d5,
        hitopanhdep_v_mb=hitopanhdep_v_mb,
        hitopanhdep_v_ab=hitopanhdep_v_ab,
        exd4=exd4,
        hitopanxwor_v_ab=hitopanxwor_v_ab,
        hitopanxwor_v_mb=hitopanxwor_v_mb,
        exd5=exd5
    )
    return res

In [18]:
calc_dhs(ogddat, pid=-1)

{'pid': -1,
 'gad_phq_v_hitop': np.float64(0.9112855644690636),
 'baars_v_hitop': np.float64(0.8123879253327188),
 'd1': np.float64(0.09889763913634486),
 'hitop_v_mab': np.float64(0.23165601409732495),
 'hitop_v_ab': np.float64(0.07872288544624717),
 'd2': np.float64(0.15293312865107778),
 'baars_v_ab': np.float64(0.11670544710082709),
 'baars_v_mab': np.float64(0.18780185880044908),
 'd3': np.float64(-0.071096411699622),
 'hitopdep_v_mb': np.float64(0.19423099984362469),
 'hitopdep_v_ab': np.float64(0.15997225130092296),
 'd4': np.float64(0.03425874854270172),
 'hitopanx_v_ab': np.float64(0.22406880343359092),
 'hitopanx_v_mb': np.float64(0.21688973024262714),
 'd5': np.float64(0.007179073190963781),
 'hitopanhdep_v_mb': np.float64(0.16996155569114269),
 'hitopanhdep_v_ab': np.float64(0.14324909373527617),
 'exd4': np.float64(0.026712461955866518),
 'hitopanxwor_v_ab': np.float64(0.23232844787092954),
 'hitopanxwor_v_mb': np.float64(0.19779823820744458),
 'exd5': np.float64(0.0345302

In [19]:
perm_res = [calc_dhs(ogddat, pid=-1)]
for pid in range(nperms):
    permed_data = pd.concat((ddat.loc[:, nonpermed_cols], ddat.loc[:, ['sample_type'] + permed_cols].groupby("sample_type").sample(frac=1, replace=False, random_state=rng).reset_index(drop=True)), axis=1)
    perm_res.append(calc_dhs(permed_data, pid=pid))
perm_res = pd.DataFrame(perm_res)

In [20]:
for dh in ['d1', 'd2', 'd3', 'd4', 'd5', 'exd4', 'exd5']:
    dhp = (perm_res.loc[0, dh] <= perm_res[dh]).mean()
    print(f"{dh}:{dhp}")

d1:0.000999000999000999
d2:0.000999000999000999
d3:0.984015984015984
d4:0.10989010989010989
d5:0.1798201798201798
exd4:0.14985014985014986
exd5:0.012987012987012988


In [ ]:
perm_res.to_pickle(convdiv_res_dir / 'divergent_perm_res.pkl')

# Do some bootstraps to get confidence intervals

In [ ]:
nboots = 1000
bootdf = []
for bid in range(nboots):
    boot_data = ddat.groupby('sample_type').sample(frac=0.8, replace=False, random_state=rng).reset_index()
    bootdf.append(calc_dhs(boot_data, pid=bid))
bootdf = pd.DataFrame(bootdf).rename(columns={'pid':'bid'})

In [ ]:
bootdf.describe(percentiles=[0.025, 0.975])

In [ ]:
calc_dhs(ogddat, pid=-1)

In [ ]:
bootdf.describe(percentiles=[0.025, 0.975]).iloc[[4,5], :]

In [ ]:
bootdf.to_pickle(convdiv_res_dir / 'divergent_boot_res.pkl')